# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya
This notebook provides a step-by-step template for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/en/latest/) library, based on a Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets and their associated fields. All entities are referenced by their `@id` as per Croissant best practices.

In [ ]:
# List available record sets and fields by `@id`
record_sets = list(dataset.record_sets.keys())
print("Available record sets (@id):")
for rsid in record_sets:
    record_set = dataset.record_sets[rsid]
    print(f"  - {rsid}")
    # List fields for this record set by @id and name
    field_ids = list(record_set.fields.keys())
    print("    Fields (@id - name):")
    for fid in field_ids:
        f = record_set.fields[fid]
        print(f"      - {fid} : {getattr(f, 'name', getattr(f, '@id', None))}")

## 3. Data Extraction
Load data from each record set into a DataFrame for further analysis. You can use the `@id` of any record set and field identified above.

In [ ]:
# Extract data from all available record sets into pandas DataFrames
dataframes = {}
for record_set_id in record_sets:
    # Use iterator to fetch up to 1000 records for demo purposes
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# For demonstration, pick the first record set for detailed work
if len(record_sets) > 0:
    demo_record_set_id = record_sets[0]
    print(f"\nFields available in primary record set '{demo_record_set_id}':\n{dataframes[demo_record_set_id].columns.tolist()}")
    display(dataframes[demo_record_set_id].head())
else:
    print("No record sets found in this Croissant dataset.")

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field from the main record set to demonstrate example cleaning and feature engineering steps, such as outlier removal, normalization, and grouping. All fields are referenced by their `@id`.

Edit the below cell to use the `@id` of a numeric field you wish to explore, as printed above.

In [ ]:
from pandas.api.types import is_numeric_dtype
# Choose which record set to analyze
record_set_id = demo_record_set_id
df = dataframes[record_set_id]

# Identify numeric fields by @id (column names)
numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col])]
print(f"Numeric fields found in record set '{record_set_id}': {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]  # Use the first numeric field for demonstration
    threshold = df[numeric_field].quantile(0.9)  # Use 90th percentile as a sample threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f} (top 10%):")
    display(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, normalized_col]].head())

    # Try grouping by a categorical field (the second column if it is not numeric)
    possible_group_fields = [col for col in df.columns if col != numeric_field and not is_numeric_dtype(df[col])]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        print(f"\nGrouping filtered data by '{group_field}' and computing mean values:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
else:
    print("No numeric fields found in this record set to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

Below, we plot the distribution of the analyzed numeric field, and if a grouping categorical field is available, we create a mean value barplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_fields:
    # Distribution plot for the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        # Barplot by group
        plt.figure(figsize=(10,4))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        sns.barplot(x=group_means.index.astype(str), y=group_means.values)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and processing the FAIR² dataset using the `mlcroissant` Python library, referencing all key dataset elements by their `@id` as per the Croissant specification. We:
- Inspected the dataset metadata and record sets;
- Extracted records using their `@id`;
- Explored available fields and performed EDA on numerics;
- Visualized field distributions and means by categorical grouping.

You can adapt this workflow to analyze any Croissant-compatible dataset. For more, visit the [mlcroissant documentation](https://mlcroissant.readthedocs.io/en/latest/).